In [1]:
"""
DAY 9: Numerical Validation of Mantle Convection

Tasks:
1. Check temperature and velocity values.
2. Calculate top and bottom Nusselt numbers.
3. Check the velocity divergence.
4. Plot validation results.
5. Examine grid and time sensitivity if available.
"""

from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 1. FIND THE RESULTS FOLDER
# ============================================================

folders = sorted(
    Path(".").glob("combined_mantle_results*"),
    key=lambda folder: folder.stat().st_mtime,
    reverse=True
)

results_folder = None

for folder in folders:
    if list(folder.glob("mantle_Ra_*.npz")):
        results_folder = folder
        break

if results_folder is None:
    raise FileNotFoundError(
        "No saved simulation files found. "
        "Run the combined mantle convection program first."
    )

output_folder = results_folder / "day9_validation"
output_folder.mkdir(exist_ok=True)

print("Input folder:", results_folder.resolve())
print("Day 9 output:", output_folder.resolve())


# ============================================================
# 2. VALIDATE EACH RAYLEIGH NUMBER
# ============================================================

summary = []

for file in sorted(results_folder.glob("mantle_Ra_*.npz")):

    data = np.load(file)

    ra = float(data["ra"])
    T = data["T"]
    u = data["u"]
    w = data["w"]
    x = data["x"]
    z = data["z"]

    dx = x[1] - x[0]
    dz = z[1] - z[0]

    print("\nChecking Ra =", ra)

    # --------------------------------------------------------
    # Temperature checks
    # --------------------------------------------------------

    temperature_min = float(np.min(T))
    temperature_max = float(np.max(T))

    temperature_finite = bool(np.isfinite(T).all())
    velocity_finite = bool(
        np.isfinite(u).all() and np.isfinite(w).all()
    )

    # --------------------------------------------------------
    # Nusselt-number estimates
    # --------------------------------------------------------

    bottom_gradient = (T[1, 1:-1] - T[0, 1:-1]) / dz
    top_gradient = (T[-1, 1:-1] - T[-2, 1:-1]) / dz

    nu_bottom = -float(np.mean(bottom_gradient))
    nu_top = -float(np.mean(top_gradient))

    mismatch = (
        100 * abs(nu_bottom - nu_top)
        / max(abs(nu_bottom), abs(nu_top), 1e-12)
    )

    # --------------------------------------------------------
    # Approximate incompressibility check
    # div(u) = du/dx + dw/dz
    # --------------------------------------------------------

    du_dx = np.gradient(u, dx, axis=1)
    dw_dz = np.gradient(w, dz, axis=0)

    divergence = du_dx + dw_dz

    divergence_rms = float(
        np.sqrt(np.mean(divergence[1:-1, 1:-1]**2))
    )

    # --------------------------------------------------------
    # Record the results
    # --------------------------------------------------------

    summary.append({
        "Ra": ra,
        "temperature_min": temperature_min,
        "temperature_max": temperature_max,
        "temperature_finite": temperature_finite,
        "velocity_finite": velocity_finite,
        "Nu_bottom": nu_bottom,
        "Nu_top": nu_top,
        "Nu_mismatch_percent": mismatch,
        "divergence_rms": divergence_rms
    })

    print("Temperature range:", temperature_min, "to", temperature_max)
    print("Bottom Nu:", nu_bottom)
    print("Top Nu:", nu_top)
    print("Nu mismatch (%):", mismatch)
    print("Divergence RMS:", divergence_rms)

    # --------------------------------------------------------
    # Flag results needing further investigation
    # --------------------------------------------------------

    if not temperature_finite or not velocity_finite:
        print("WARNING: Non-finite values detected.")

    if temperature_min < -0.05 or temperature_max > 1.05:
        print("WARNING: Temperature is outside the approximate [0, 1] range.")

    if mismatch > 10:
        print("WARNING: Top and bottom heat fluxes differ by over 10%.")

    # --------------------------------------------------------
    # Plot the saved temperature field
    # --------------------------------------------------------

    X, Z = np.meshgrid(x, z)

    plt.figure(figsize=(7, 5))

    contour = plt.contourf(
        X, Z, T, levels=21, cmap="inferno"
    )

    plt.colorbar(contour, label="Dimensionless temperature")
    plt.xlabel("x / d")
    plt.ylabel("z / d")
    plt.title(f"Day 9: Temperature Validation, Ra = {ra:g}")
    plt.tight_layout()

    plt.savefig(
        output_folder / f"temperature_check_Ra_{ra:g}.png",
        dpi=180
    )

    plt.close()


# ============================================================
# 3. SAVE VALIDATION TABLE
# ============================================================

if summary:

    csv_file = output_folder / "day9_validation_summary.csv"

    with open(csv_file, "w", newline="", encoding="utf-8") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=list(summary[0].keys())
        )

        writer.writeheader()
        writer.writerows(summary)

    print("\nValidation table saved:", csv_file)


# ============================================================
# 4. PLOT NUSSELT-NUMBER COMPARISON
# ============================================================

if summary:

    summary = sorted(summary, key=lambda item: item["Ra"])

    ra_values = [item["Ra"] for item in summary]
    nu_bottom = [item["Nu_bottom"] for item in summary]
    nu_top = [item["Nu_top"] for item in summary]

    plt.figure(figsize=(7, 5))

    plt.plot(
        ra_values, nu_bottom, "o-",
        label="Bottom Nu estimate"
    )

    plt.plot(
        ra_values, nu_top, "s--",
        label="Top Nu estimate"
    )

    plt.axhline(
        1, color="black", linestyle=":",
        label="Conduction reference (Nu = 1)"
    )

    plt.xlabel("Rayleigh number, Ra")
    plt.ylabel("Nusselt-number estimate")
    plt.title("Day 9: Heat-Flux Consistency Check")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        output_folder / "day9_nusselt_validation.png",
        dpi=180
    )

    plt.close()


# ============================================================
# 5. PLOT TEMPERATURE RANGE
# ============================================================

if summary:

    ra_values = [item["Ra"] for item in summary]
    t_min = [item["temperature_min"] for item in summary]
    t_max = [item["temperature_max"] for item in summary]

    plt.figure(figsize=(7, 5))

    plt.plot(ra_values, t_min, "o-", label="Minimum temperature")
    plt.plot(ra_values, t_max, "s-", label="Maximum temperature")

    plt.axhline(0, color="black", linestyle=":")
    plt.axhline(1, color="black", linestyle=":")

    plt.xlabel("Rayleigh number, Ra")
    plt.ylabel("Dimensionless temperature")
    plt.title("Day 9: Temperature Range Check")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        output_folder / "day9_temperature_range.png",
        dpi=180
    )

    plt.close()


# ============================================================
# 6. CHECK GRID/TIME SENSITIVITY DATA IF AVAILABLE
# ============================================================

sensitivity_file = results_folder / "grid_time_sensitivity.csv"

if sensitivity_file.exists():

    print("\nExisting grid/time sensitivity data found.")

    with open(
        sensitivity_file, newline="", encoding="utf-8"
    ) as f:

        sensitivity_rows = list(csv.DictReader(f))

    print("Number of sensitivity records:", len(sensitivity_rows))
    print("File:", sensitivity_file)

else:

    print(
        "\nNo grid_time_sensitivity.csv found. "
        "Run the combined program with sensitivity tests enabled."
    )


print("\n" + "=" * 55)
print("DAY 9 VALIDATION FINISHED")
print("Check the day9_validation folder for the CSV and plots.")
print("Remember: diagnostic checks do not prove full convergence.")
print("=" * 55)

Input folder: C:\Users\athil\OneDrive\Desktop\combined_mantle_results_2
Day 9 output: C:\Users\athil\OneDrive\Desktop\combined_mantle_results_2\day9_validation

Checking Ra = 1000.0
Temperature range: 0.0 to 1.0
Bottom Nu: 5.836153095734757
Top Nu: 7.763581696574158
Nu mismatch (%): 24.826538525252055
Divergence RMS: 1.4315010626459518e-13

Checking Ra = 2500.0
Temperature range: 0.0 to 1.0
Bottom Nu: 9.746131680905368
Top Nu: 12.454051976431867
Nu mismatch (%): 21.74328725021371
Divergence RMS: 3.1221982537956443e-13

Checking Ra = 500.0
Temperature range: 0.0 to 1.0
Bottom Nu: 7.0535586228536165
Top Nu: 7.6511121496079335
Nu mismatch (%): 7.81002179905228
Divergence RMS: 7.312926794296519e-14

Checking Ra = 5000.0
Temperature range: 0.0 to 1.0
Bottom Nu: 13.92769140082633
Top Nu: 14.250779182114124
Nu mismatch (%): 2.2671587087202627
Divergence RMS: 5.2766168309471e-13

Validation table saved: combined_mantle_results_2\day9_validation\day9_validation_summary.csv

Existing grid/time s